# 🏋️ Namakan — Reusable Fine-Tuning Notebook

**LoRA/QLoRA Fine-Tuning for Any Model, Any Dataset**

---

## Section 0: Client Parameters — FILL THESE IN

Everything you need to configure is in this cell. All other cells are universal.

In [ ]:
# ============================================================
# CLIENT PARAMETERS — Edit these before running
# ============================================================

CLIENT_NAME = "[CLIENT_NAME]"           # e.g. "Mayo Clinic"
BASE_MODEL = "[BASE_MODEL]"             # HuggingFace model ID e.g. "Qwen/Qwen2.5-7B-Instruct"
HF_TOKEN = "[HF_TOKEN]"                 # HuggingFace token (needed for gated models)

# Data source — URL to JSONL file (GCS signed URL, GitHub raw, or direct URL)
DATA_URL = "[DATA_URL]"                 # e.g. "https://storage.googleapis.com/bucket/data/train.jsonl"
DATA_FORMAT = "chat"                    # "chat" (instruction/output) or "plain" ({"text": "..."})

# Model persona
SYSTEM_PROMPT = "[SYSTEM_PROMPT]"       # e.g. "You are a helpful customer service assistant."

# Training config
MAX_SEQ_LENGTH = 512
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
EPOCHS = 3
LEARNING_RATE = 2e-4
BATCH_SIZE = 2
GRAD_ACCUM = 8
VAL_SPLIT = 0.1                        # Fraction of data held out for validation

# Early stopping
EARLY_STOPPING_PATIENCE = 2

# Output
OUTPUT_DIR = "/content/adapters"

# Validation
assert CLIENT_NAME != "[CLIENT_NAME]", "⚠️ Fill in CLIENT_NAME in Section 0"
assert BASE_MODEL != "[BASE_MODEL]", "⚠️ Fill in BASE_MODEL in Section 0"
assert HF_TOKEN != "[HF_TOKEN]", "⚠️ Fill in HF_TOKEN in Section 0"
assert DATA_URL != "[DATA_URL]", "⚠️ Fill in DATA_URL in Section 0"
assert SYSTEM_PROMPT != "[SYSTEM_PROMPT]", "⚠️ Fill in SYSTEM_PROMPT in Section 0"

print(f"✅ Client: {CLIENT_NAME}")
print(f"✅ Model: {BASE_MODEL}")
print(f"✅ Data: {DATA_URL}")
print(f"✅ System: {SYSTEM_PROMPT[:60]}...")
print(f"✅ LoRA R={LORA_R}, α={LORA_ALPHA}, epochs={EPOCHS}, lr={LEARNING_RATE}")

---

## Section 1: Setup — Install Dependencies

In [ ]:
!pip install --quiet transformers accelerate bitsandbytes peft trl
!pip install --quiet huggingface_hub datasets scipy torch
!pip install --quiet torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
print("✅ Dependencies installed!")

## Section 2: GPU Check

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"VRAM: {vram_gb:.1f} GB")
    if vram_gb < 8:
        print("⚠️  Warning: < 8GB VRAM. Consider using smaller batch size or 4-bit quantization.")

## Section 3: Load Base Model + Tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print(f"Loading tokenizer from {BASE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token

print(f"Loading model {BASE_MODEL}...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
)
print("✅ Model loaded!")

## Section 4: Attach LoRA Adapters

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("✅ LoRA adapters attached!")

## Section 5: Load + Process Training Data

In [ ]:
import requests
import io
from datasets import Dataset

def load_jsonl(url):
    """Load JSONL from URL (supports GCS signed URLs, GitHub raw, any direct URL)"""
    response = requests.get(url)
    response.raise_for_status()
    lines = response.text.strip().split('\n')
    return [json.loads(line) for line in lines if line.strip()]

print(f"Downloading data from {DATA_URL}...")
raw_data = load_jsonl(DATA_URL)
print(f"Loaded {len(raw_data)} examples")

def process_chat_example(example):
    """Process instruction/output pairs into chat format."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Task: " + example.get("instruction", "")},
        {"role": "assistant", "content": example.get("output", "")}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    tokens = tokenizer(
        text,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
        return_tensors="np"
    )
    return {
        "input_ids": tokens["input_ids"][0].tolist(),
        "attention_mask": tokens["attention_mask"][0].tolist(),
        "labels": tokens["input_ids"][0].tolist(),
    }

def process_plain_example(example):
    """Process plain text examples."""
    tokens = tokenizer(
        example.get("text", ""),
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
        return_tensors="np"
    )
    return {
        "input_ids": tokens["input_ids"][0].tolist(),
        "attention_mask": tokens["attention_mask"][0].tolist(),
        "labels": tokens["input_ids"][0].tolist(),
    }

print(f"Processing as {DATA_FORMAT} format...")
process_fn = process_chat_example if DATA_FORMAT == "chat" else process_plain_example
processed = [process_fn(ex) for ex in raw_data]

full_dataset = Dataset.from_list(processed)
full_dataset = full_dataset.train_test_split(test_size=VAL_SPLIT, shuffle=True, seed=42)
train_ds = full_dataset["train"]
val_ds = full_dataset["test"]
print(f"✅ Train: {len(train_ds)} | Val: {len(val_ds)}")

## Section 6: Live Training Charts

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from IPython import display
import numpy as np

plt.ion()

# Layout: Train Loss | Val Loss | Table | Stats
fig = plt.figure(figsize=(20, 7))
gs = fig.add_gridspec(1, 4, wspace=0.35, width_ratios=[1.2, 1.2, 1, 0.8])

ax_train = fig.add_subplot(gs[0])
ax_val = fig.add_subplot(gs[1])
ax_table = fig.add_subplot(gs[2])
ax_stats = fig.add_subplot(gs[3])

# Initialize styling
ax_train.set_facecolor('#e6f0ff')
ax_train.set_xlabel('Step')
ax_train.set_ylabel('Loss')
ax_train.set_title('Training Loss')
ax_train.grid(True, alpha=0.3)

ax_val.set_facecolor('#ffe6e6')
ax_val.set_xlabel('Step')
ax_val.set_ylabel('Loss')
ax_val.set_title('Validation Loss')
ax_val.grid(True, alpha=0.3)

ax_table.axis('off')
ax_table.set_title('Recent Steps', fontsize=11, fontweight='bold')
ax_stats.axis('off')

# Live data storage
all_steps, all_losses = [], []
all_val_steps, all_val_losses = [], []

display.display(fig)

def update_charts(step, loss, eval_loss):
    global all_steps, all_losses, all_val_steps, all_val_losses

    if step is None:
        return

    # Guard: skip None/NaN
    if loss is not None and isinstance(loss, (int, float)) and not np.isnan(float(loss)):
        all_steps.append(step)
        all_losses.append(float(loss))

    if eval_loss is not None and isinstance(eval_loss, (int, float)) and not np.isnan(float(eval_loss)):
        all_val_steps.append(step)
        all_val_losses.append(float(eval_loss))

    # Training Loss Chart
    ax_train.clear()
    ax_train.set_facecolor('#e6f0ff')
    if all_steps and all_losses:
        ax_train.plot(all_steps, all_losses, 'b-', linewidth=2, marker='o', markersize=4)
        ax_train.annotate('{:.4f}'.format(all_losses[-1]),
                         xy=(all_steps[-1], all_losses[-1]),
                         xytext=(5, 5), textcoords='offset points', fontsize=10, color='blue')
    ax_train.set_xlabel('Step')
    ax_train.set_ylabel('Loss')
    ax_train.set_title('Training Loss (Step {})'.format(step))
    ax_train.legend(['Train Loss'], loc='upper right')
    ax_train.grid(True, alpha=0.3)

    # Validation Loss Chart
    ax_val.clear()
    ax_val.set_facecolor('#ffe6e6')
    if all_val_steps and all_val_losses:
        ax_val.plot(all_val_steps, all_val_losses, 'r-', linewidth=2, marker='s', markersize=6)
        ax_val.annotate('{:.4f}'.format(all_val_losses[-1]),
                        xy=(all_val_steps[-1], all_val_losses[-1]),
                        xytext=(5, 5), textcoords='offset points', fontsize=10, color='red')
    ax_val.set_xlabel('Step')
    ax_val.set_ylabel('Loss')
    ax_val.set_title('Validation Loss (Step {})'.format(step))
    ax_val.legend(['Val Loss'], loc='upper right')
    ax_val.grid(True, alpha=0.3)

    # Recent Steps Table
    ax_table.clear()
    ax_table.axis('off')
    col_labels = ['Step', 'Train', 'Val']
    start_idx = max(0, len(all_steps) - 15)
    table_data = []
    for j in range(start_idx, len(all_steps)):
        loss_val = '{:.4f}'.format(all_losses[j])
        val_val = '-'
        if all_steps[j] in all_val_steps:
            val_idx = all_val_steps.index(all_steps[j])
            val_val = '{:.4f}'.format(all_val_losses[val_idx])
        table_data.append([all_steps[j], loss_val, val_val])

    if table_data:
        table = ax_table.table(cellText=table_data, colLabels=col_labels,
                               loc='center', cellLoc='center', colWidths=[0.3, 0.35, 0.35])
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1, 1.5)
        for j in range(3):
            table[(0, j)].set_facecolor('#2E75B6')
            table[(0, j)].set_text_props(color='white', fontweight='bold')
        for row_idx in range(1, len(table_data) + 1):
            color = '#D6DCE4' if row_idx % 2 == 0 else '#FFFFFF'
            for col_idx in range(3):
                table[(row_idx, col_idx)].set_facecolor(color)
    ax_table.set_title('Recent Steps', fontsize=11, fontweight='bold')

    # Stats panel
    ax_stats.clear()
    ax_stats.axis('off')
    loss_str = '{:.4f}'.format(loss) if (loss is not None and isinstance(loss, (int, float))) else 'N/A'
    val_str = '{:.4f}'.format(eval_loss) if (eval_loss is not None and isinstance(eval_loss, (int, float))) else 'N/A'
    stats_text = (
        f'{CLIENT_NAME}\n'
        '==================\n'
        f'Step:       {step:>6}\n\n'
        f'Train Loss: {loss_str:>10}\n'
        f'Val Loss:   {val_str:>10}\n\n'
        f'Model: {BASE_MODEL.split("/")[-1]}\n'
        f'LoRA R={LORA_R}, α={LORA_ALPHA}\n'
        f'E: {EPOCHS} | LR: {LEARNING_RATE}'
    )
    ax_stats.text(0.5, 0.5, stats_text, fontsize=12, family='monospace',
                  ha='center', va='center', transform=ax_stats.transAxes,
                  bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    display.clear_output(wait=True)
    display.display(fig)

print("Charts ready!")


## Section 7: Train

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from transformers import EarlyStoppingCallback, TrainerCallback

class LiveChartCallback(TrainerCallback):
    def __init__(self, update_fn):
        self.update_fn = update_fn
        self.last_eval_loss = None
        self.last_train_loss = None

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return
        step = state.global_step
        loss = logs.get('loss')
        eval_loss = logs.get('eval_loss')

        if loss is not None:
            self.last_train_loss = loss
        if eval_loss is not None:
            self.last_eval_loss = eval_loss
        else:
            eval_loss = self.last_eval_loss

        self.update_fn(step, self.last_train_loss, eval_loss)


training_args = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/{CLIENT_NAME}",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_steps=10,
    fp16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    weight_decay=0.1,
    max_grad_norm=0.3,
    seed=42,
    remove_unused_columns=False,
)

data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE),
        LiveChartCallback(update_charts)
    ],
)

print("=" * 60)
print("TRAINING CONFIG")
print("=" * 60)
print(f"Client: {CLIENT_NAME}")
print(f"Model: {BASE_MODEL}")
print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")
print(f"Epochs: {EPOCHS} | LR: {LEARNING_RATE} | Batch: {BATCH_SIZE}x{GRAD_ACCUM}")
print(f"LoRA R={LORA_R}, α={LORA_ALPHA}")
print(f"Optimizer: paged_adamw_8bit | Scheduler: cosine")
print(f"Max grad norm: 0.3 | Early stop patience: {EARLY_STOPPING_PATIENCE}")
print("=" * 60)
print()
print("Starting training...")
trainer.train()
display.clear_output()
print("Training complete!")
if trainer.state.best_metric:
    print("Best eval loss: {:.4f}".format(trainer.state.best_metric))


## Section 8: Final Evaluation

In [ ]:
import math

print("=" * 60)
print("FINAL EVALUATION")
print("=" * 60)

eval_results = trainer.evaluate()
final_loss = eval_results['eval_loss']
final_ppl = math.exp(final_loss)

print(f"Final Val Loss: {final_loss:.4f}")
print(f"Final Val Perplexity: {final_ppl:.2f}")

best_checkpoint = trainer.state.best_model_checkpoint
print(f"Best checkpoint: {best_checkpoint}")

## Section 9: Final Training Charts

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

history = trainer.state.log_history

train_steps = [h["step"] for h in history if "loss" in h and "eval_loss" not in h]
train_losses = [h["loss"] for h in history if "loss" in h and "eval_loss" not in h]
val_steps = [h["step"] for h in history if "eval_loss" in h]
val_losses = [h["eval_loss"] for h in history if "eval_loss" in h]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1, ax2 = axes

ax1.plot(train_steps, train_losses, 'b-', linewidth=2, label='Train Loss')
ax1.set_xlabel('Step')
ax1.set_ylabel('Loss')
ax1.set_title(f'{CLIENT_NAME} — Training Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(val_steps, val_losses, 'r-', linewidth=2, label='Val Loss')
ax2.set_xlabel('Step')
ax2.set_ylabel('Loss')
ax2.set_title(f'{CLIENT_NAME} — Validation Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📊 Best val loss: {min(val_losses):.4f} at step {val_steps[val_losses.index(min(val_losses))]}")

## Section 10: Test the Model

In [ ]:
model.eval()

# Sample test prompts
test_prompts = [
    "What can you help me with?",
    "Tell me about your capabilities.",
]

print("Testing trained model...\n")
for test_instruction in test_prompts:
    print(f"Input: {test_instruction}")
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Task: {test_instruction}"}
    ]
    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.1,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = result.split("assistant\n")[-1].strip()
    print(f"Output: {response}")
    print("-" * 60)

## Section 11: Save Adapter

In [ ]:
import os

save_dir = f"{OUTPUT_DIR}/{CLIENT_NAME}/final"
os.makedirs(save_dir, exist_ok=True)

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print(f"✅ Model saved to: {save_dir}")

# Also save best checkpoint
if best_checkpoint:
    print(f"   Best checkpoint: {best_checkpoint}")

# Mount Google Drive and backup
try:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_dir = f"/content/drive/MyDrive/namakan/adapters/{CLIENT_NAME}"
    os.makedirs(drive_dir, exist_ok=True)
    import shutil
    shutil.copytree(save_dir, f"{drive_dir}/final", dirs_exist_ok=True)
    print(f"✅ Backed up to Google Drive: {drive_dir}")
except Exception as e:
    print(f"⚠️  Could not mount Google Drive: {e}")
    print("   Download the adapter manually from the Colab file browser.")

---

## Next Steps

1. **Download adapter** from `/content/adapters/{CLIENT_NAME}/final/`
2. **Deploy** to inference stack (see namakan-technical/pipeline/fine-tuned-models/workflows/deployment_pipeline.py)
3. **Share notebook** with client as reproducibility proof
4. **Log metrics**: Final val loss, perplexity, best checkpoint step

---
*Namakan Fine-Tuning Pipeline — {BASE_MODEL} for {CLIENT_NAME}*